# Классические модели

Исходные Colab (копии оригиналов):
- ["ru_sentiment_pipeline_adapted_gzip_robust_ipynb".ipynb](https://colab.research.google.com/github/SamDerden/SpeechSense/blob/main/%22ru_sentiment_pipeline_adapted_gzip_robust_ipynb%22.ipynb)
- [Копия_блокнота_"ru_sentiment_pipeline_adapted_gzip_robust_ipynb".ipynb](https://colab.research.google.com/github/SamDerden/SpeechSense/blob/main/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22ru_sentiment_pipeline_adapted_gzip_robust_ipynb%22.ipynb)
- [Трансформеры.ipynb](https://colab.research.google.com/github/SamDerden/SpeechSense/blob/main/%D0%A2%D1%80%D0%B0%D0%BD%D1%81%D1%84%D0%BE%D1%80%D0%BC%D0%B5%D1%80%D1%8B.ipynb)

Перенос исходного кода; выполнение и метрики пока не проверены. Исторические результаты в оригиналах.

In [ ]:
from pathlib import Path
import os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'download_dataset.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Clone SpeechSense and run from its root or notebooks directory')
os.chdir(ROOT)
DATA_PATH = ROOT / 'data' / 'merged_sentiment_dataset.csv.gz'
if not DATA_PATH.exists():
    raise FileNotFoundError('Run python data/download_dataset.py first')


In [ ]:
!pip install -q gdown lightgbm optuna plotly kaleido pymorphy3

# fastText (если ещё нужен из части I)
try:
    import fasttext
except ImportError:
    !pip install -q setuptools wheel pybind11
    !pip install -q git+https://github.com/facebookresearch/fastText.git

In [ ]:
import os, re, time, random, copy, warnings
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from functools import lru_cache
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sp
import gdown

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.decomposition import TruncatedSVD
from sklearn.naive_bayes import MultinomialNB, ComplementNB, BernoulliNB
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, learning_curve,
)
from sklearn.metrics import (
    f1_score, accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_fscore_support,
)
from sklearn.preprocessing import normalize

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.max_colwidth", 150)
sns.set_style("whitegrid")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- Классы и их маппинг ----
CLASS_MAP   = {0: "Нейтральный", 1: "Позитивный", 2: "Негативный"}
CLASS_ORDER = [0, 1, 2]
CLASS_NAMES = [CLASS_MAP[i] for i in CLASS_ORDER]

# Алиасы для совместимости с кодом части I (коллега) и части II (наш)
LABELS      = CLASS_NAMES
LABELS_RU   = CLASS_NAMES
id2label    = CLASS_MAP
label2id    = {v: k for k, v in CLASS_MAP.items()}

# ---- Прочее ----
LEADERBOARD = []
RANDOM_STATE = 42

def report(name, y_true, y_pred, train_time=0.0, group="—"):
    """Считает macro-F1 и accuracy, добавляет строку в leaderboard."""
    from sklearn.metrics import f1_score, accuracy_score
    f1 = f1_score(y_true, y_pred, average="macro")
    acc = accuracy_score(y_true, y_pred)
    LEADERBOARD.append({"группа": group, "модель": name,
                        "macro-F1": round(f1, 4), "accuracy": round(acc, 4),
                        "обучение, с": round(train_time, 1)})
    print(f"{name:55s}  macro-F1={f1:.4f}  acc={acc:.4f}  ({train_time:.1f} с)")
    return f1

# Проверка, что всё на месте
assert LABELS == ["Нейтральный", "Позитивный", "Негативный"]
assert label2id["Позитивный"] == 1
print("Константы классов:")
print(f"  CLASS_MAP   = {CLASS_MAP}")
print(f"  LABELS      = {LABELS}")
print(f"  label2id    = {label2id}")

In [ ]:
import pandas as pd
df = pd.read_csv(DATA_PATH, compression='gzip')

In [ ]:
required = {"text", "sentiment", "split"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Отсутствуют обязательные колонки: {missing}")

# Убеждаемся, что sentiment — числа 0/1/2
df["sentiment"] = pd.to_numeric(df["sentiment"], errors="coerce")
df = df.dropna(subset=["sentiment", "text", "split"]).copy()
df["sentiment"] = df["sentiment"].astype(int)

uniq = sorted(df["sentiment"].unique())
print(f"Уникальные sentiment: {uniq}")
assert set(uniq).issubset({0, 1, 2}), f"Неизвестные значения: {uniq}"

df["sentiment_name"] = df["sentiment"].map(CLASS_MAP)
assert df["sentiment_name"].isnull().sum() == 0

print("\n=== Итоговое распределение классов ===")
counts = df["sentiment_name"].value_counts().reindex(LABELS)
shares = (counts / counts.sum() * 100).round(2)
print(pd.DataFrame({"count": counts, "share, %": shares}))

print("\n=== Разбиение ===")
print(df["split"].value_counts())

if "label_source" in df.columns:
    print("\n=== label_source × split ===")
    print(pd.crosstab(df["label_source"], df["split"], margins=True))

In [ ]:
# Для последующего EDA используем названия классов на русском.
# Поле text — единый источник текста для train и validation.
df["full_text"] = df["text"].fillna("").astype(str).str.strip()
df["sentiment_name"] = df["sentiment"].map(CLASS_MAP)

print("Пример full_text:")
print(df.loc[df["full_text"].str.len().gt(0), "full_text"].iloc[0][:500])

if "source_dataset" in df.columns:
    print("\nИсточники данных:")
    display(pd.crosstab(df["source_dataset"], df["split"], margins=True))
if "label_source" in df.columns:
    print("\nИсточники разметки:")
    display(pd.crosstab(df["label_source"], df["split"], margins=True))


In [ ]:
import re

URL_RE     = re.compile(r"https?://\S+|www\.\S+")
HTML_RE    = re.compile(r"<[^>]+>")
MENTION_RE = re.compile(r"[@#]\w+")
EMAIL_RE   = re.compile(r"\S+@\S+")

# Диапазоны Unicode, покрывающие эмодзи
_EMOJI_CLASS = (
    "\U0001F300-\U0001FAFF"
    "\U00002600-\U000027BF"
    "\U0001F1E0-\U0001F1FF"
    "\U0000FE00-\U0000FE0F"
    "\U00002B00-\U00002BFF"
)
EMOJI_RE = re.compile("[" + _EMOJI_CLASS + "]", flags=re.UNICODE)

# ВАЖНО: та же эмодзи-класс добавлена и в токенизатор,
# иначе findall их выкидывает.
TOKEN_RE = re.compile(
    r"[а-яёa-z0-9]+(?:-[а-яёa-z0-9]+)*"
    r"|[!?]+"
    r"|[()]+"
    r"|[" + _EMOJI_CLASS + r"]",
    flags=re.UNICODE,
)


def clean_text(text: str,
               keep_emoji: bool = True,
               mask_numbers: bool = False,
               fix_yo: bool = True,
               max_len: int = 10_000) -> str:
    if not isinstance(text, str):
        return ""
    text = text[:max_len]

    text = HTML_RE.sub(" ", text)
    text = URL_RE.sub(" URL ", text)
    text = EMAIL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = re.sub(r"[\n\r\t]+", " ", text)

    if keep_emoji:
        text = EMOJI_RE.sub(lambda m: f" {m.group(0)} ", text)
    else:
        text = EMOJI_RE.sub(" ", text)

    text = text.lower()
    if fix_yo:
        text = text.replace("ё", "е")

    if mask_numbers:
        text = re.sub(r"\d+([.,]\d+)?", " NUM ", text)

    text = re.sub(
        r"[^а-яa-z0-9\s!?()" + _EMOJI_CLASS + r"]",
        " ", text,
    )
    return re.sub(r"\s+", " ", text).strip()


def tokenize(text: str) -> str:
    if not isinstance(text, str) or not text:
        return ""
    tokens = TOKEN_RE.findall(text)
    out = []
    for t in tokens:
        if len(t) > 1 and t[0] in "!?()":
            t = t[0] * 2          # «!!!!» → «!!»
        out.append(t)
    return " ".join(out)


# --- Проверка ---
demo = "Ужасное качество!!! ((( пришло за 30 дней(( размер XL не подошёл 😡😡 А цена 1 999 руб!"
print("Оригинал:", demo)
print("clean  :", clean_text(demo))
print("tok    :", tokenize(clean_text(demo)))
print()
print("Ожидание: в tok ДОЛЖНЫ быть 😡 😡 (по одному на каждый эмодзи)")

In [ ]:
print(f"Размер ДО очистки: {df.shape[0]:,}")

df_clean = df.copy()
df_clean = df_clean.dropna(subset=["sentiment", "split"])
df_clean["full_text"] = df_clean["full_text"].fillna("").astype(str)

t0 = time.time()
df_clean["text_clean"] = df_clean["full_text"].apply(
    lambda t: tokenize(clean_text(t, keep_emoji=True, fix_yo=True, max_len=10_000))
)
print(f"Предобработка: {time.time() - t0:.1f} с")

MIN_LEN = 10
before = len(df_clean)
df_clean = df_clean[df_clean["text_clean"].str.len() >= MIN_LEN].copy()
df_clean = df_clean.drop_duplicates(subset=["split", "text_clean"]).reset_index(drop=True)
df_clean["sentiment_name"] = df_clean["sentiment"].map(CLASS_MAP)

print(f"Удалено (пустые, короткие, дубли): {before - len(df_clean):,}")
print(f"Размер ПОСЛЕ: {len(df_clean):,}")
print("\nРазбиение:")
print(df_clean["split"].value_counts())

---
## Train / Validation split

Используем готовое разбиение из CSV: `train` для обучения, `validation` для оценки. Validation не участвует ни в выборе обучающих примеров, ни в обучении векторизаторов.

In [ ]:
# Для полного эксперимента используем всю train-часть.
# Для более быстрого пробного запуска можно установить USE_FULL_TRAIN = False.
USE_FULL_TRAIN = True
MAX_TRAIN_SAMPLES = 60_000

available_splits = set(df_clean["split"].astype(str).str.lower().unique())
if {"train", "validation"}.issubset(available_splits):
    train_df = df_clean[df_clean["split"].astype(str).str.lower().eq("train")].copy()
    test_df = df_clean[df_clean["split"].astype(str).str.lower().eq("validation")].copy()
    print("Используется исходное разбиение из CSV: train / validation.")
else:
    # Резервный вариант на случай, если в файле нет готового validation.
    print("Внимание: train/validation не найдены. Создаю stratified train/test split 80/20.")
    train_df, test_df = train_test_split(
        df_clean, test_size=0.2, random_state=RANDOM_STATE,
        stratify=df_clean["sentiment"]
    )

if not USE_FULL_TRAIN and len(train_df) > MAX_TRAIN_SAMPLES:
    n_classes = train_df["sentiment"].nunique()
    samples_per_class = max(1, MAX_TRAIN_SAMPLES // n_classes)
    train_df = (
        train_df.groupby("sentiment", group_keys=False)
        .apply(lambda group: group.sample(
            n=min(len(group), samples_per_class), random_state=RANDOM_STATE
        ))
        .reset_index(drop=True)
    )
    print(f"Для ускорения взята стратифицированная подвыборка train: {len(train_df):,} строк.")
else:
    print(f"Используем train целиком: {len(train_df):,} строк.")

if train_df["sentiment"].nunique() < 2 or test_df["sentiment"].nunique() < 2:
    raise ValueError("В train или validation недостаточно классов для обучения и оценки.")

print("\nTrain labels:")
print(train_df["sentiment"].map(CLASS_MAP).value_counts())
print("\nValidation labels:")
print(test_df["sentiment"].map(CLASS_MAP).value_counts())


In [ ]:
X_train = train_df["text_clean"]
y_train = train_df["sentiment"]
X_test = test_df["text_clean"]
y_test = test_df["sentiment"]

print(f"Train: {len(X_train):,} | Validation: {len(X_test):,}")
print("Распределение в train:")
print(y_train.value_counts().sort_index().rename(index=CLASS_MAP))
print("\nРаспределение в validation:")
print(y_test.value_counts().sort_index().rename(index=CLASS_MAP))


---
## 2–3. Векторизации + классические модели

Обучаем несколько комбинаций «векторизатор + классификатор» на одном и том же train/test.


In [ ]:
# ---------- Модели ----------
models = {
    "MultinomialNB (BoW 1-2)": Pipeline([
        ("vect", CountVectorizer(ngram_range=(1, 2), min_df=3, max_features=100_000)),
        ("clf", MultinomialNB())
    ]),

    "BernoulliNB (бинарный BoW 1-2)": Pipeline([
        ("vect", CountVectorizer(ngram_range=(1, 2), binary=True, min_df=3, max_features=100_000)),
        ("clf", BernoulliNB())
    ]),

    "ComplementNB (TF-IDF 1-2)": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=100_000)),
        ("clf", ComplementNB())
    ]),

    "LogReg (TF-IDF 1-2)": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=100_000)),
        ("clf", LogisticRegression(max_iter=1000, C=1.0, n_jobs=-1, random_state=RANDOM_STATE))
    ]),

    "LinearSVC (TF-IDF 1-2)": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=100_000)),
        ("clf", LinearSVC(C=1.0, dual="auto", random_state=RANDOM_STATE, max_iter=2000))
    ]),

    "LinearSVC (word+char)": Pipeline([
        ("features", FeatureUnion([
            ("word", TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=50_000)),
            ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_features=30_000))
        ])),
        ("clf", LinearSVC(C=0.5, dual="auto", random_state=RANDOM_STATE, max_iter=2000))
    ]),
}

print(f"Будет обучено {len(models)} моделей\n")


In [ ]:
LEADERBOARD.clear()  # не дублируем строки, если обучение запускается повторно

# ---------- Обучение и оценка ----------
for name, model in models.items():
    print(f"Обучаю: {name} ...", end=" ", flush=True)
    t0 = time.time()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    elapsed = time.time() - t0
    report(name, y_test, y_pred, elapsed)
    print()


---
## 4. Итоговое сравнение


In [ ]:
df_lb = pd.DataFrame(LEADERBOARD).sort_values("macro-F1", ascending=False).reset_index(drop=True)

styled = (
    df_lb.style
    .format({
        "macro-F1": "{:.4f}",
        "accuracy": "{:.4f}",
        "обучение, с": "{:.1f}"
    })
    .highlight_max(subset=["macro-F1", "accuracy"], color="#d4edda")
    .set_caption("Leaderboard: macro-F1 и Accuracy на тестовой выборке")
    .set_table_styles([
        {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "15px"), ("font-weight", "bold")]},
        {"selector": "th", "props": [("text-align", "center"), ("background-color", "#f8f9fa")]},
        {"selector": "td", "props": [("text-align", "center")]}
    ])
    .hide(axis="index")
)

display(styled)


In [ ]:
# Барплот для наглядности
fig, ax = plt.subplots(figsize=(10, 5))
colors = sns.color_palette("viridis", len(df_lb))
bars = ax.barh(df_lb["модель"], df_lb["macro-F1"], color=colors)
ax.set_xlabel("macro-F1")
ax.set_title("Сравнение моделей по macro-F1")
ax.set_xlim(0, 1)
for bar, val in zip(bars, df_lb["macro-F1"]):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2, f"{val:.4f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()


### 4.1 Подробный отчёт лучшей модели


In [ ]:
# Берём лучшую модель по macro-F1 на validation
if not LEADERBOARD:
    raise RuntimeError("Сначала выполните ячейку обучения моделей (ячейка 23).")

df_lb = pd.DataFrame(LEADERBOARD).sort_values("macro-F1", ascending=False).reset_index(drop=True)
best_name = df_lb.iloc[0]["модель"]
best_model = models[best_name]

print(f"Лучшая модель: {best_name}")
print(f"macro-F1 = {df_lb.iloc[0]['macro-F1']:.4f}, accuracy = {df_lb.iloc[0]['accuracy']:.4f}\n")

y_pred_best = best_model.predict(X_test)
print(classification_report(
    y_test, y_pred_best,
    labels=CLASS_ORDER,
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
))


In [ ]:
# Матрица ошибок на validation
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred_best, labels=CLASS_ORDER)
disp = ConfusionMatrixDisplay(cm, display_labels=["Нейтр.", "Поз.", "Нег."])
disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
ax.set_title(f"Матрица ошибок — {best_name}")
plt.tight_layout()
plt.show()


### Как интерпретировать матрицу ошибок

Строки матрицы соответствуют истинной тональности, столбцы — предсказанной. Значения на диагонали — правильно классифицированные примеры; значения вне диагонали показывают, какие классы модель путает между собой. Конкретные числа формируются при запуске ноутбука и зависят от обученных моделей.

---
## Краткие выводы по фактическому запуску

Таблица и вывод ниже формируются по результатам текущего запуска на validation из предоставленного CSV. Фиксированные метрики из исходного ноутбука удалены, потому что они относятся к другому разбиению данных.


In [ ]:
# Метрики формируются динамически после обучения моделей.
df_lb = pd.DataFrame(LEADERBOARD).sort_values("macro-F1", ascending=False).reset_index(drop=True)
if df_lb.empty:
    print("Результатов пока нет. Сначала выполните ячейку обучения моделей.")
else:
    display(df_lb)
    best = df_lb.iloc[0]
    print(f"Лучшая модель по macro-F1: {best['модель']}")
    print(f"macro-F1: {best['macro-F1']:.4f}; accuracy: {best['accuracy']:.4f}")
    if len(df_lb) > 1:
        print(f"Разница macro-F1 с моделью на втором месте: {best['macro-F1'] - df_lb.iloc[1]['macro-F1']:.4f}")

print("\nУчитывайте, что train содержит несколько источников, включая 27 136 примеров с эвристической разметкой по рейтингу, "
      "а validation сформирован из исходного ru_sentiment_dataset_validation.")


## Optuna и fastText supervised

---
# Часть II. Углублённые эксперименты: Optuna, нейросеть, ансамбли и анализ ошибок

## Что делается в этой части

1. Лемматизация.
2. Расширенные векторизации: BoW, TF-IDF, sublinear, char n-граммы, word+char.
3. **Optuna (TPE)** для LogReg, LinearSVC, SGD, **LightGBM**.
4. **Нейросеть** (PyTorch MLP) — обязательный компонент ансамбля.
5. **Дополнительные парадигмы**: kNN, ComplementNB, fastText supervised.
6. **Матрица корреляции ошибок** — диагностика разнообразия.
7. **Ансамбли**: soft voting, weighted average, stacking.
8. **Проверки на переобучение** (learning curves, кривая C).
9. **Анализ ошибок**: confusion matrix, длина, уверенность, спорная разметка.
10. Финальный leaderboard и выводы.

Лемматизация не делается — на русской тональности она в пределах шума, а char_wb 3-5 и fastText уже ловят морфологию через n-граммы символов.


In [ ]:
# ============================================================
# Часть II. Импорты, конфиг, утилиты
# ============================================================
import os, time, pickle, random, warnings, subprocess, gc
from pathlib import Path
from functools import lru_cache
from collections import Counter

import numpy as np
import pandas as pd
import scipy.sparse as sp

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (f1_score, accuracy_score,
                             classification_report, confusion_matrix,
                             precision_recall_fscore_support)

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# --- Безопасные fallbacks ---
SEED        = globals().get("SEED", 42)
CLASS_MAP   = globals().get("CLASS_MAP",   {0: "Нейтральный", 1: "Позитивный", 2: "Негативный"})
CLASS_ORDER = globals().get("CLASS_ORDER", [0, 1, 2])
CLASS_NAMES = globals().get("CLASS_NAMES", [CLASS_MAP[i] for i in CLASS_ORDER])
LABEL_TO_ID = {v: k for k, v in CLASS_MAP.items()}
LEADERBOARD = globals().get("LEADERBOARD", [])

# ============================================================
# Единый конфиг
# ============================================================
CONFIG = {
    # --- Общий TF-IDF для сравнимых экспериментов ---
    "min_df": 3,
    "max_features": 100_000,

    # --- Optuna LogReg ---
    "optuna_trials": 15,
    "optuna_sub": 30_000,
    "cv_folds": 3,

    # --- LightGBM ---
    "lgb_sub": 60_000,
    "lgb_svd_dim": 150,
    "lgb_optuna_trials": 12,

    # --- fastText ---
    "ft_epochs": 10,
    "ft_dim": 100,
    "ft_lr": 0.5,

    # --- CNN ---
    "cnn_sub": 60_000,
    "cnn_epochs": 3,
    "cnn_max_vocab": 50_000,
    "cnn_min_freq": 2,
    "cnn_max_len": 64,
    "cnn_emb_dim": 100,
}

print("CONFIG:", CONFIG)


def report(name, y_true, y_pred, train_time=0.0, group="—"):
    """Считает macro-F1 и accuracy, добавляет строку в LEADERBOARD."""
    f1  = f1_score(y_true, y_pred, average="macro")
    acc = accuracy_score(y_true, y_pred)
    LEADERBOARD.append({
        "группа": group, "модель": name,
        "macro-F1": round(float(f1), 4),
        "accuracy": round(float(acc), 4),
        "обучение, с": round(float(train_time), 1),
    })
    print(f"{name:50s}  macro-F1={f1:.4f}  acc={acc:.4f}  ({train_time:.1f} с)")
    return f1


for _name in ["train_df", "test_df", "y_train", "y_test"]:
    assert _name in globals(), f"Не найдено: {_name} (выполните Part I)"
assert "text_clean" in train_df.columns, "train_df должен содержать text_clean"
assert "text_clean" in test_df.columns,  "test_df должен содержать text_clean"
print(f"\nTrain: {len(train_df):,} | Validation: {len(test_df):,}")
print(f"Классы: {CLASS_NAMES}")

In [ ]:
# ============================================================
# Baseline из Part I: переносим в LEADERBOARD, чтобы в финале
# видеть, какие модели Part II улучшили результат, а какие нет.
# ============================================================
BASELINE_PART_I = [
    ("1. классика", "MultinomialNB (BoW 1-2)",   0.6483, 0.6497),
    ("1. классика", "BernoulliNB (BoW 1-2)",     0.5903, 0.5963),
    ("1. классика", "ComplementNB (TF-IDF 1-2)", 0.6660, 0.6665),
    ("1. классика", "LogReg (TF-IDF 1-2)",       0.7472, 0.7592),
    ("1. классика", "LinearSVC (TF-IDF 1-2)",    0.7338, 0.7470),
    ("1. классика", "LinearSVC (word+char)",     0.7460, 0.7578),
]

LEADERBOARD.clear()
for grp, name, f1, acc in BASELINE_PART_I:
    LEADERBOARD.append({
        "группа": grp, "модель": name,
        "macro-F1": f1, "accuracy": acc,
        "обучение, с": None,
    })

BEST_BASELINE_F1 = max(r["macro-F1"] for r in LEADERBOARD)
BEST_BASELINE_NAME = max(LEADERBOARD, key=lambda r: r["macro-F1"])["модель"]

print(f"Baseline Part I: {len(LEADERBOARD)} моделей")
for row in LEADERBOARD:
    print(f"  {row['модель']:35s}  macro-F1={row['macro-F1']:.4f}")
print(f"\nЛучший baseline: {BEST_BASELINE_NAME} = {BEST_BASELINE_F1:.4f}")
print("Цель Part II: побить это значение.")

## 2. Optuna (TPE) для LogReg

В Part I LogReg обучалась с фиксированным `C=1.0`, `ngram_range=(1,2)`,
`min_df=3`. Подбираем эти гиперпараметры байесовски.

- Метрика — `f1_macro` на StratifiedKFold (3 фолда) по подвыборке 30k.
- `MedianPruner` убивает плохие trials на первом фолде.
- Solver — `liblinear`: единственный стабильный и с L1, и с L2 на
  многоклассовой задаче. На 200k × 100k с `n_jobs=-1` не ускоряется,
  поэтому CV идёт в один поток.
- 15 trials × 3 фолда × 30k — около 10–15 минут.

In [ ]:
# ============================================================
# Optuna для LogReg на подвыборке 30k
# ============================================================
sub_idx = train_df.sample(CONFIG["optuna_sub"], random_state=SEED).index
X_opt = train_df.loc[sub_idx, "text_clean"]
y_opt = y_train.loc[sub_idx].values


def objective_logreg(trial):
    C         = trial.suggest_float("C", 1e-2, 1e1, log=True)
    penalty   = trial.suggest_categorical("penalty", ["l1", "l2"])
    sublinear = trial.suggest_categorical("sublinear_tf", [True, False])
    ngram_max = trial.suggest_int("ngram_max", 1, 2)
    min_df    = trial.suggest_int("min_df", 3, 10)
    cw        = trial.suggest_categorical("class_weight", [None, "balanced"])

    pipe = Pipeline([
        ("v", TfidfVectorizer(
            token_pattern=r"\S+", ngram_range=(1, ngram_max),
            min_df=min_df, sublinear_tf=sublinear,
            max_features=CONFIG["max_features"],
        )),
        ("c", LogisticRegression(
            C=C, penalty=penalty, solver="liblinear",
            class_weight=cw, max_iter=1000,
            random_state=SEED, n_jobs=-1,
        )),
    ])
    cv = StratifiedKFold(n_splits=CONFIG["cv_folds"],
                         shuffle=True, random_state=SEED)
    return cross_val_score(pipe, X_opt, y_opt, cv=cv,
                           scoring="f1_macro", n_jobs=1).mean()


study_logreg = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=SEED),
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=1),
    study_name="logreg_tfidf",
)
study_logreg.optimize(objective_logreg,
                      n_trials=CONFIG["optuna_trials"],
                      show_progress_bar=True)

print(f"\nLogReg best CV F1 = {study_logreg.best_value:.4f}")
print(f"Best params: {study_logreg.best_params}")

In [ ]:
# ============================================================
# Лучший LogReg (по Optuna) на полном train
# ============================================================
p = study_logreg.best_params

vec_opt = TfidfVectorizer(
    token_pattern=r"\S+", ngram_range=(1, p["ngram_max"]),
    min_df=p["min_df"], sublinear_tf=p["sublinear_tf"],
    max_features=CONFIG["max_features"],
)
X_tr_opt = vec_opt.fit_transform(train_df["text_clean"])
X_va_opt = vec_opt.transform(test_df["text_clean"])

clf_opt = LogisticRegression(
    C=p["C"], penalty=p["penalty"], solver="liblinear",
    class_weight=p["class_weight"], max_iter=1000,
    random_state=SEED, n_jobs=-1,
)
t0 = time.perf_counter()
clf_opt.fit(X_tr_opt, y_train)
tt = time.perf_counter() - t0

y_pred_opt  = clf_opt.predict(X_va_opt)
y_proba_opt = clf_opt.predict_proba(X_va_opt)
report("LogReg (Optuna)", y_test, y_pred_opt, tt, group="2. Optuna")

## 4. fastText supervised

fastText supervised =
мешок эмбеддингов слов **и биграмм** + линейный слой + softmax.
Плюс символьные n-граммы (`minn`, `maxn`) — устойчивость к опечаткам
и словоформам.

Обучение — секунды на CPU. Сравниваем 3 конфигурации:

1. Только униграммы — базовый вариант.
2. + биграммы слов.
3. + биграммы + символьные n-граммы 3–5.

In [ ]:
# ============================================================
# fastText supervised
# ============================================================
try:
    import fasttext
    HAS_FASTTEXT = True
except ImportError:
    print("Устанавливаю fasttext-wheel...")
    subprocess.run(["pip", "install", "-q", "fasttext-wheel"], check=True)
    import fasttext
    HAS_FASTTEXT = True


def _write_ft(path, texts, ys):
    with open(path, "w", encoding="utf-8") as f:
        for t, y in zip(texts, ys):
            f.write(f"__label__{CLASS_MAP[int(y)]} {t}\n")


def ft_predict(model, texts):
    labels, _ = model.predict(list(texts))
    return np.array([LABEL_TO_ID[l[0].replace("__label__", "")] for l in labels])


def ft_predict_proba(model, texts):
    labels, probs = model.predict(list(texts), k=len(CLASS_ORDER))
    out = np.zeros((len(texts), len(CLASS_ORDER)), dtype=np.float32)
    for i, (ls, ps) in enumerate(zip(labels, probs)):
        for l, p in zip(ls, ps):
            cid = LABEL_TO_ID[l.replace("__label__", "")]
            out[i, cid] = p
    return out


ft_models = {}
if HAS_FASTTEXT:
    _write_ft("ft_train.txt", train_df["text_clean"].values, y_train.values)

    ft_configs = {
        "fastText sup (unigram)":        dict(wordNgrams=1),
        "fastText sup (+bigram)":        dict(wordNgrams=2),
        "fastText sup (+bigram +char)":  dict(wordNgrams=2, minn=3, maxn=5),
    }
    for name, extra in ft_configs.items():
        t0 = time.perf_counter()
        model = fasttext.train_supervised(
            "ft_train.txt",
            epoch=CONFIG["ft_epochs"], lr=CONFIG["ft_lr"],
            dim=CONFIG["ft_dim"], loss="softmax",
            seed=SEED, thread=4, verbose=0, **extra,
        )
        tt = time.perf_counter() - t0
        ft_models[name] = model
        y_pred_ft = ft_predict(model, test_df["text_clean"].values)
        report(name, y_test, y_pred_ft, tt, group="4. fastText")

## Альтернативная историческая конфигурация из исследования трансформеров (отдельный split; не сравнивать напрямую)

In [ ]:
print("КЛАССИЧЕСКИЕ МОДЕЛИ")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.calibration import CalibratedClassifierCV

# Подготовка текстов
text_col = 'text_clean' if 'text_clean' in train_df.columns else 'text'

X_train_text = train_df[text_col].fillna('').astype(str).tolist()
X_test_text = test_df[text_col].fillna('').astype(str).tolist()
y_train = train_df['sentiment'].values
y_test = test_df['sentiment'].values

# Векторизация
tfidf = TfidfVectorizer(
    max_features=15000, ngram_range=(1, 2),
    min_df=5, max_df=0.95, sublinear_tf=True
)
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

# Модели
models_classic = {
    'LogReg': LogisticRegression(C=0.1, max_iter=1000, class_weight='balanced'),
    'SVM': CalibratedClassifierCV(LinearSVC(C=0.1, class_weight='balanced', max_iter=2000)),
    'NB': MultinomialNB(alpha=1.0),
}

results_classic = {}
for name, model in models_classic.items():
    print(f"\n Обучение {name}...")
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
    results_classic[name] = {'accuracy': acc, 'macro_f1': f1}

    print(f"   Accuracy: {acc:.4f} | Macro-F1: {f1:.4f}")

# Лучшая классическая модель
best_classic_name = max(results_classic, key=lambda k: results_classic[k]['macro_f1'])
best_classic_f1 = results_classic[best_classic_name]['macro_f1']
print(f"\n Лучшая классическая: {best_classic_name} (Macro-F1={best_classic_f1:.4f})")